In [ ]:
"""
export_aqi_data_v5.py
Exports aggregated AQI data from PostgreSQL to aqi_data.json.

Changes from v4:
  - Added city_pollutant_overview: city-level pollutant exceedance rates
    Shape: { "CN": { "Beijing": { "2019": { "pm25": 82.3, "pm10": 45.1, ... } } } }
    O3 excluded — over_who_limit flag not computed for O3 in source data.
    Enables bar chart to show city-level data when a city is selected,
    matching the same behavior as the line chart.

Changes from v3:
  - Added region_pm25_monthly: region + world monthly PM2.5 time series
    Built in Python from pm25_monthly (no new SQL query needed).
    Shape: { "Asia": { "2019": { "01": 68.4, ... } }, "__world__": { ... }, ... }
    Q1 2022 nulled same as country/city level.
    Fixes line chart blanking out when region or World is selected.

Changes from v2:
  - Added city_pm25_monthly: city-level monthly PM2.5 time series

Changes from v1:
  - Added city-level annual PM2.5 summary
  - Added geographic region rollups + world aggregate
  - Q1 2022 months explicitly nulled in all monthly outputs

Run once locally, commit aqi_data.json to docs/ folder.
"""

import os
import json
import psycopg2
from datetime import datetime

# ── EDIT THIS ─────────────────────────────────────────────────────
DB_CONFIG = {
    "dbname":   os.environ.get("DB_NAME", "pm25"),
    "user":     os.environ.get("DB_USER", "postgres"),
    "password": os.environ.get("DB_PASSWORD"),
    "host":     os.environ.get("DB_HOST", "localhost"),
    "port":     int(os.environ.get("DB_PORT", 5432))
}
# ──────────────────────────────────────────────────────────────────

AQI_POLLUTANTS = ["pm25", "pm10", "co", "no2", "o3", "so2"]

# O3 excluded from city pollutant overview — over_who_limit flag not computed for O3
CITY_POLLUTANTS = ["pm25", "pm10", "co", "no2", "so2"]

WHO_LIMITS = {
    "pm25": 15,
    "pm10": 45,
    "no2":  10,
    "o3":   60,
    "co":   4,
    "so2":  40
}

REGIONS = {
    "North America": ["CA", "MX", "US", "SV"],
    "South America": ["AR", "BR", "CL", "CO", "PE"],
    "Europe":        ["AT", "BA", "BG", "CH", "CY", "DE", "EE", "ES", "FI",
                      "FR", "GB", "HR", "IE", "IL", "IT", "KZ", "LT", "MK",
                      "NL", "NO", "PL", "RE", "RO", "RS", "RU", "SE", "TR", "XK"],
    "Asia":          ["BD", "CN", "HK", "ID", "IN", "JP", "KR", "KW", "MN",
                      "NP", "SG", "TH", "TW", "VN"],
    "Oceania":       ["AU", "NZ"],
    "Africa":        ["ZA"],
}

Q1_2022 = {("2022", "01"), ("2022", "02"), ("2022", "03")}


def get_connection():
    return psycopg2.connect(**DB_CONFIG)


def get_countries_and_years(conn):
    cur = conn.cursor()
    cur.execute("""
        SELECT DISTINCT country, EXTRACT(YEAR FROM date)::int AS year
        FROM aqi_full_report
        WHERE variable = 'pm25'
        ORDER BY country, year
    """)
    rows = cur.fetchall()
    countries = sorted(set(r[0] for r in rows))
    years     = sorted(set(r[1] for r in rows))
    return countries, years


def get_pollutant_overview(conn):
    cur = conn.cursor()
    cur.execute("""
        SELECT
            country,
            EXTRACT(YEAR FROM date)::int AS year,
            variable,
            AVG(monthly_percent_exceeded)::float AS avg_pct
        FROM aqi_full_report
        WHERE variable = ANY(%s)
          AND monthly_percent_exceeded IS NOT NULL
        GROUP BY country, year, variable
        ORDER BY country, year, variable
    """, (AQI_POLLUTANTS,))

    result = {}
    for country, year, variable, pct in cur.fetchall():
        year = str(year)
        result.setdefault(country, {}).setdefault(year, {})[variable] = (
            round(pct, 4) if pct is not None else None
        )
    return result


def get_city_pollutant_overview(conn):
    """
    City-level pollutant exceedance rates by year.
    O3 excluded — over_who_limit flag was not computed for O3 in source data.
    Shape: { "CN": { "Beijing": { "2019": { "pm25": 82.3, "pm10": 45.1, ... } } } }
    """
    cur = conn.cursor()
    cur.execute("""
        SELECT
            country,
            city,
            EXTRACT(YEAR FROM date)::int AS year,
            variable,
            AVG(monthly_percent_exceeded)::float AS avg_pct
        FROM aqi_full_report
        WHERE variable = ANY(%s)
          AND monthly_percent_exceeded IS NOT NULL
          AND over_who_limit IS NOT NULL
        GROUP BY country, city, year, variable
        ORDER BY country, city, year, variable
    """, (CITY_POLLUTANTS,))

    result = {}
    for country, city, year, variable, pct in cur.fetchall():
        year = str(int(year))
        result \
            .setdefault(country, {}) \
            .setdefault(city, {}) \
            .setdefault(year, {})[variable] = round(float(pct), 4) if pct is not None else None
    return result


def get_pm25_monthly(conn):
    """Country-level monthly PM2.5. Q1 2022 forced to None."""
    cur = conn.cursor()
    cur.execute("""
        SELECT
            country,
            EXTRACT(YEAR FROM date)::int  AS year,
            EXTRACT(MONTH FROM date)::int AS month,
            AVG("30_day_rolling_avg")::float AS avg_rolling
        FROM aqi_full_report
        WHERE variable = 'pm25'
          AND "30_day_rolling_avg" IS NOT NULL
          AND "30_day_rolling_avg" > 0
        GROUP BY country, year, month
        ORDER BY country, year, month
    """)

    result = {}
    for country, year, month, avg in cur.fetchall():
        year_str  = str(int(year))
        month_str = str(int(month)).zfill(2)
        value = None if (year_str, month_str) in Q1_2022 else (
            round(avg, 2) if avg is not None else None
        )
        result.setdefault(country, {}).setdefault(year_str, {})[month_str] = value
    return result


def get_pm25_annual(conn):
    """Country/year annual PM2.5 summary for left panel."""
    cur = conn.cursor()
    cur.execute("""
        SELECT
            country,
            EXTRACT(YEAR FROM date)::int AS year,
            AVG("30_day_rolling_avg")::float    AS annual_avg,
            AVG(magnitude_score)::float          AS avg_magnitude,
            AVG(CASE WHEN over_who_limit = 1
                     THEN 1.0 ELSE 0.0 END)::float AS pct_days_exceeded
        FROM aqi_full_report
        WHERE variable = 'pm25'
          AND "30_day_rolling_avg" IS NOT NULL
          AND "30_day_rolling_avg" > 0
        GROUP BY country, year
        ORDER BY country, year
    """)

    result = {}
    for country, year, annual_avg, avg_magnitude, pct_days in cur.fetchall():
        year = str(year)
        result.setdefault(country, {})[year] = {
            "avg":               round(annual_avg, 2)    if annual_avg    is not None else None,
            "magnitude_score":   round(avg_magnitude, 2) if avg_magnitude is not None else None,
            "pct_days_exceeded": round(pct_days, 4)      if pct_days      is not None else None,
        }
    return result


def get_city_pm25_annual(conn):
    """City/country/year annual PM2.5 summary."""
    cur = conn.cursor()
    cur.execute("""
        SELECT
            country,
            city,
            EXTRACT(YEAR FROM date)::int AS year,
            AVG("30_day_rolling_avg")::float AS annual_avg,
            AVG(magnitude_score)::float      AS avg_magnitude
        FROM aqi_full_report
        WHERE variable = 'pm25'
          AND "30_day_rolling_avg" IS NOT NULL
          AND "30_day_rolling_avg" > 0
        GROUP BY country, city, year
        ORDER BY country, city, year
    """)

    result = {}
    for country, city, year, annual_avg, avg_magnitude in cur.fetchall():
        year = str(year)
        result.setdefault(country, {}).setdefault(city, {})[year] = {
            "avg":             round(annual_avg, 2)    if annual_avg    is not None else None,
            "magnitude_score": round(avg_magnitude, 2) if avg_magnitude is not None else None,
        }
    return result


def get_city_pm25_monthly(conn):
    """
    City-level monthly PM2.5 30-day rolling average.
    Q1 2022 forced to None — same completeness rule as country-level.
    Shape: { "CN": { "Beijing": { "2019": { "01": 124.3, ... } } } }
    """
    cur = conn.cursor()
    cur.execute("""
        SELECT
            country,
            city,
            EXTRACT(YEAR FROM date)::int  AS year,
            EXTRACT(MONTH FROM date)::int AS month,
            AVG("30_day_rolling_avg")::float AS avg_rolling
        FROM aqi_full_report
        WHERE variable = 'pm25'
          AND "30_day_rolling_avg" IS NOT NULL
          AND "30_day_rolling_avg" > 0
        GROUP BY country, city, year, month
        ORDER BY country, city, year, month
    """)

    result = {}
    for country, city, year, month, avg in cur.fetchall():
        year_str  = str(int(year))
        month_str = str(int(month)).zfill(2)
        value = None if (year_str, month_str) in Q1_2022 else (
            round(avg, 2) if avg is not None else None
        )
        result.setdefault(country, {}).setdefault(city, {}).setdefault(year_str, {})[month_str] = value
    return result


def build_region_rollups(pollutant_overview, pm25_annual):
    """Region and world aggregates from country data."""
    all_years = sorted({
        year
        for country_data in pm25_annual.values()
        for year in country_data.keys()
    })

    def avg_or_none(values):
        vals = [v for v in values if v is not None]
        return round(sum(vals) / len(vals), 4) if vals else None

    all_regions = dict(REGIONS)
    all_regions["__world__"] = list({c for cs in REGIONS.values() for c in cs})

    region_overview = {}
    region_annual   = {}

    for region, countries in all_regions.items():
        region_overview[region] = {}
        region_annual[region]   = {}

        for year in all_years:
            region_overview[region][year] = {}
            for p in AQI_POLLUTANTS:
                vals = [pollutant_overview.get(c, {}).get(year, {}).get(p) for c in countries]
                region_overview[region][year][p] = avg_or_none(vals)

            avgs = [pm25_annual.get(c, {}).get(year, {}).get("avg")               for c in countries]
            mags = [pm25_annual.get(c, {}).get(year, {}).get("magnitude_score")   for c in countries]
            pcts = [pm25_annual.get(c, {}).get(year, {}).get("pct_days_exceeded") for c in countries]
            region_annual[region][year] = {
                "avg":               avg_or_none(avgs),
                "magnitude_score":   avg_or_none(mags),
                "pct_days_exceeded": avg_or_none(pcts),
            }

    return region_overview, region_annual


def build_region_pm25_monthly(pm25_monthly):
    """
    Build region + world monthly PM2.5 time series from country-level monthly data.
    No new SQL — aggregates the already-fetched pm25_monthly dict in Python.

    Method: for each region/year/month, average the country values that have data.
    Q1 2022 propagates as None automatically (country values are already None there).

    Shape: { "Asia": { "2019": { "01": 68.4, "02": 71.2, ... }, ... }, "__world__": {...}, ... }
    """
    def avg_or_none(values):
        vals = [v for v in values if v is not None]
        return round(sum(vals) / len(vals), 2) if vals else None

    all_regions = dict(REGIONS)
    all_regions["__world__"] = list({c for cs in REGIONS.values() for c in cs})

    all_year_months = set()
    for country_years in pm25_monthly.values():
        for year, months in country_years.items():
            for month in months:
                all_year_months.add((year, month))

    result = {}
    for region, countries in all_regions.items():
        result[region] = {}
        years = sorted({ym[0] for ym in all_year_months})
        for year in years:
            months = sorted({ym[1] for ym in all_year_months if ym[0] == year})
            result[region][year] = {}
            for month in months:
                vals = [
                    pm25_monthly.get(c, {}).get(year, {}).get(month)
                    for c in countries
                ]
                result[region][year][month] = avg_or_none(vals)

    return result


def get_city_list(city_pm25_annual):
    return {country: sorted(cities.keys()) for country, cities in city_pm25_annual.items()}


def main():
    print("Connecting to PostgreSQL...")
    conn = get_connection()

    print("Fetching countries and years...")
    countries, years = get_countries_and_years(conn)
    print(f"  {len(countries)} countries, years: {years}")

    print("Fetching pollutant overview (country-level)...")
    pollutant_overview = get_pollutant_overview(conn)

    print("Fetching city-level pollutant overview...")
    city_pollutant_overview = get_city_pollutant_overview(conn)
    total_city_pol = sum(len(cities) for cities in city_pollutant_overview.values())
    print(f"  {total_city_pol} cities with pollutant data across {len(city_pollutant_overview)} countries")

    print("Fetching PM2.5 monthly (country-level)...")
    pm25_monthly = get_pm25_monthly(conn)

    print("Fetching PM2.5 annual (country-level)...")
    pm25_annual = get_pm25_annual(conn)

    print("Fetching city-level PM2.5 annual summary...")
    city_pm25_annual = get_city_pm25_annual(conn)
    city_list = get_city_list(city_pm25_annual)
    total_cities = sum(len(v) for v in city_list.values())
    print(f"  {total_cities} cities across {len(city_list)} countries")

    print("Fetching city-level PM2.5 monthly time series...")
    city_pm25_monthly = get_city_pm25_monthly(conn)
    total_series = sum(len(cities) for cities in city_pm25_monthly.values())
    print(f"  {total_series} city time series exported")

    conn.close()

    print("Building region and world rollups...")
    region_overview, region_annual = build_region_rollups(pollutant_overview, pm25_annual)

    print("Building region monthly time series...")
    region_pm25_monthly = build_region_pm25_monthly(pm25_monthly)
    print(f"  {len(region_pm25_monthly)} regions built: {list(region_pm25_monthly.keys())}")

    output = {
        "meta": {
            "generated":   datetime.now().isoformat(),
            "years":       years,
            "countries":   countries,
            "regions":     list(REGIONS.keys()),
            "who_limits":  WHO_LIMITS,
            "note_2022":   "Q1 2022 excluded — completeness below 75% EPA threshold",
            "note_o3":     "O3 exceedance flag not computed — requires 8-hour averaging methodology not applied in this dataset.",
        },
        "pollutant_overview":      pollutant_overview,
        "city_pollutant_overview": city_pollutant_overview,
        "pm25_monthly":            pm25_monthly,
        "pm25_annual":             pm25_annual,
        "city_list":               city_list,
        "city_pm25_annual":        city_pm25_annual,
        "city_pm25_monthly":       city_pm25_monthly,
        "region_overview":         region_overview,
        "region_annual":           region_annual,
        "region_pm25_monthly":     region_pm25_monthly,
    }

    output_path = "aqi_data.json"
    with open(output_path, "w") as f:
        json.dump(output, f, indent=2)

    raw_size = len(json.dumps(output))
    print(f"\nDone. Written to {output_path}")
    print(f"File size: {raw_size // 1024} KB")
    print("\nSpot-check:")
    print(f"  CN/2021 pm25_annual:              {pm25_annual.get('CN',{}).get('2021')}")
    print(f"  Asia/2021 region_annual:          {region_annual.get('Asia',{}).get('2021')}")
    print(f"  World/2021 region_annual:         {region_annual.get('__world__',{}).get('2021')}")
    print(f"  CN cities (first 5):              {city_list.get('CN',[])[:5]}")
    bj_pol_2021 = city_pollutant_overview.get('CN',{}).get('Beijing',{}).get('2021')
    print(f"  Beijing/2021 pollutant overview:  {bj_pol_2021}")
    bj_2021 = dict(list((city_pm25_monthly.get('CN',{}).get('Beijing',{}).get('2021',{})).items())[:3])
    bj_2022 = dict(list((city_pm25_monthly.get('CN',{}).get('Beijing',{}).get('2022',{})).items())[:3])
    print(f"  Beijing/2021 Jan-Mar monthly:     {bj_2021}")
    print(f"  Beijing/2022 Q1 (→None):          {bj_2022}")
    asia_2021 = dict(list((region_pm25_monthly.get('Asia',{}).get('2021',{})).items())[:3])
    world_2022_q1 = dict(list((region_pm25_monthly.get('__world__',{}).get('2022',{})).items())[:3])
    print(f"  Asia/2021 Jan-Mar monthly:        {asia_2021}")
    print(f"  World/2022 Q1 (→None):            {world_2022_q1}")


if __name__ == "__main__":
    main()

Connecting to PostgreSQL...
Fetching countries and years...
  54 countries, years: [2019, 2020, 2021, 2022, 2023]
Fetching pollutant overview (country-level)...
Fetching city-level pollutant overview...
  426 cities with pollutant data across 54 countries
Fetching PM2.5 monthly (country-level)...
Fetching PM2.5 annual (country-level)...
Fetching city-level PM2.5 annual summary...
  408 cities across 54 countries
Fetching city-level PM2.5 monthly time series...
  408 city time series exported
Building region and world rollups...
Building region monthly time series...
  7 regions built: ['North America', 'South America', 'Europe', 'Asia', 'Oceania', 'Africa', '__world__']

Done. Written to aqi_data.json
File size: 624 KB

Spot-check:
  CN/2021 pm25_annual:              {'avg': 84.62, 'magnitude_score': 84.41, 'pct_days_exceeded': 0.9943}
  Asia/2021 region_annual:          {'avg': 80.8043, 'magnitude_score': 82.82, 'pct_days_exceeded': 0.9906}
  World/2021 region_annual:         {'avg': 

In [ ]:
import psycopg2
try:
    conn = psycopg2.connect(
        dbname="pm25", 
        user="postgres", 
        password="password", 
        host="127.0.0.1", 
        connect_timeout=5  # This prevents the infinite spinning circle!
    )
    print("Success!")
    conn.close()
except Exception as e:
    print(f"Connection failed: {e}")

In [ ]:
print("Hello from WSL")